# 🧠 Traductor Shiwilu ↔ Español con EncoderDecoderModel (Colab Ready)
Este notebook entrena un modelo de traducción usando Hugging Face Transformers y un tokenizer BPE entrenado desde cero con `tokenizers`. Es compatible con lenguas de escasos recursos.

✅ Compatible con GPU/CPU
✅ Sin conflictos de protobuf
✅ Ideal para Colab

In [1]:
!pip install -U transformers datasets tokenizers sacrebleu protobuf==3.20.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Uninstalling protobuf-5.29.4:
      Successfully uninstalled protobuf-5.29.4
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.2
    Uninstalling transf

## 📁 Subir archivos `shiwilu.txt` y `espanol.txt`

In [3]:
# Combinar datos en un solo archivo limpio
with open("shiwilu_sintetico.txt", encoding="utf-8") as f1, open("espanol_sintetico.txt", encoding="utf-8") as f2:
    lines = [line.strip() for line in f1] + [line.strip() for line in f2]

with open("combined_sintetico.txt", "w", encoding="utf-8") as f:
    for line in lines:
        f.write(line + "\n")

In [4]:
# Entrenar tokenizer BPE
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

tokenizer = Tokenizer(models.BPE(unk_token='[UNK]'))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=500, special_tokens=['[PAD]', '[UNK]', '[BOS]', '[EOS]'])
tokenizer.train(["combined_sintetico.txt"], trainer)
tokenizer.save("tokenizer_sintetico.json")

In [5]:
# Cargar tokenizer
from transformers import PreTrainedTokenizerFast
tok = PreTrainedTokenizerFast(
    tokenizer_file="tokenizer_sintetico.json",
    bos_token='[BOS]', eos_token='[EOS]', pad_token='[PAD]', unk_token='[UNK]'
)
tok.model_max_length = 128

In [6]:
# Crear dataset paralelo
from datasets import Dataset
with open("shiwilu_sintetico.txt", encoding="utf-8") as f1, open("espanol_sintetico.txt", encoding="utf-8") as f2:
    dataset = Dataset.from_dict({
        "shw": [line.strip() for line in f1],
        "es": [line.strip() for line in f2],
    }).train_test_split(test_size=0.1)

In [7]:
# Preprocesar dataset
def preprocess(example):
    inputs = tok(example['shw'], padding='max_length', truncation=True, max_length=128)
    targets = tok(example['es'], padding='max_length', truncation=True, max_length=128).input_ids
    inputs['labels'] = targets
    return inputs

tokenized_dataset = dataset.map(preprocess, remove_columns=['shw', 'es'])

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [17]:
tokenized_dataset['train']

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 900
})

In [18]:
# Modelo encoder-decoder
from transformers import EncoderDecoderModel
model = EncoderDecoderModel.from_encoder_decoder_pretrained("bert-base-uncased", "bert-base-uncased")
model.config.pad_token_id = tok.pad_token_id
model.config.decoder_start_token_id = tok.bos_token_id
model.config.eos_token_id = tok.eos_token_id

Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.crossattention.self.key.bias', 'bert.e

In [9]:
# Entrenamiento
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch
from transformers import DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./shiwilu_model",
    do_train=True,
    do_eval=True,
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available()
)


data_collator = DataCollatorForSeq2Seq(tokenizer=tok, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)
# trainer.train()

In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hector-gomez (hector-gomez-pontificia-universidad-cat-lica-del-per-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:577: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


Step,Training Loss
500,0.474700
1000,0.048600
1500,0.034000
2000,0.021500
2500,0.023000
3000,0.018300
3500,0.016600
4000,0.014900
4500,0.014800
5000,0.015300


/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:577: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:577: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no 

TrainOutput(global_step=11250, training_loss=0.0379134367412991, metrics={'train_runtime': 1587.0941, 'train_samples_per_second': 28.354, 'train_steps_per_second': 7.088, 'total_flos': 6901358630400000.0, 'train_loss': 0.0379134367412991, 'epoch': 50.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/shiwilu_traductor_model")
tok.save_pretrained("/content/drive/MyDrive/shiwilu_traductor_model")



Mounted at /content/drive


('/content/drive/MyDrive/shiwilu_traductor_model/tokenizer_config.json',
 '/content/drive/MyDrive/shiwilu_traductor_model/special_tokens_map.json',
 '/content/drive/MyDrive/shiwilu_traductor_model/tokenizer.json')

In [ ]:
model = EncoderDecoderModel.from_pretrained("/content/drive/MyDrive/shiwilu_traductor_model")
tok = PreTrainedTokenizerFast.from_pretrained("/content/drive/MyDrive/shiwilu_traductor_model")


In [ ]:
print("pad_token_id:", model.config.pad_token_id)
print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("eos_token_id:", model.config.eos_token_id)

pad_token_id: 0
decoder_start_token_id: 2
eos_token_id: 3


## 📏 Evaluación del Modelo con BLEU y Ejemplos de Traducción

In [ ]:
# Traducción y métricas
from sacrebleu import corpus_bleu

def traducir(texto):
    inputs = tok(texto, return_tensors="pt", padding=True, truncation=True).to(model.device)

    output = model.generate(
        **inputs,
        decoder_start_token_id=model.config.decoder_start_token_id,
        max_length=128
    )


    return tok.decode(output[0], skip_special_tokens=True)

In [ ]:
refs, hyps = [], []
for ex in dataset['test']:
    ref = ex['es']
    hyp = traducir(ex['shw'])
    refs.append([ref])
    hyps.append(hyp)

bleu = corpus_bleu(hyps, list(zip(*refs)))
exact = sum([1 for r, h in zip(refs, hyps) if r[0].strip() == h.strip()]) / len(refs)

print(f"BLEU score: {bleu.score:.2f}")
print(f"Exact Match: {exact*100:.2f}%")

BLEU score: 79.63
Exact Match: 0.00%


### 🔍 Ejemplos de traducción generados por el modelo


In [ ]:
from sacrebleu.metrics import BLEU
import random

bleu_metric = BLEU()
resultados = []

# Tomar 10 ejemplos aleatorios
indices = random.sample(range(len(dataset['test'])), 10)

for i in indices:
    entrada = dataset['test'][i]['shw']
    referencia = dataset['test'][i]['es']
    traduccion = traducir(entrada)

    # Calcular BLEU entre traducción y referencia
    score = bleu_metric.sentence_score(traduccion, [referencia]).score

    resultados.append((score, entrada, referencia, traduccion))

# Ordenar por mejor puntuación BLEU descendente
resultados.sort(reverse=True)

# Mostrar los mejores
for score, entrada, referencia, traduccion in resultados:
    print(f"🎖️ BLEU: {score:.2f}")
    print(f"🔸 Entrada:     {entrada}")
    print(f"🎯 Referencia: {referencia}")
    print(f"🤖 Traducción: {traduccion}\n")


🎖️ BLEU: 100.00
🔸 Entrada:     KU' ÑUK LLINSERPI ALA'SA' KERKA'
🎯 Referencia: yo lee un libro.
🤖 Traducción: yo lee un libro .

🎖️ BLEU: 100.00
🔸 Entrada:     KU' ÑUK LLINSERPI ALA'SA' KERKA'
🎯 Referencia: yo lee un libro.
🤖 Traducción: yo lee un libro .

🎖️ BLEU: 100.00
🔸 Entrada:     KU' KUDA TULU'NER'LLI TANKU
🎯 Referencia: nosotros camina por el bosque.
🤖 Traducción: nosotros camina por el bosque .

🎖️ BLEU: 80.91
🔸 Entrada:     KU' ÑUK KU'LA TULU'NER'LLI TANKU
🎯 Referencia: yo no camina por el bosque.
🤖 Traducción: nosotros no camina por el bosque .

🎖️ BLEU: 80.91
🔸 Entrada:     KU' APER KU'LA TULU'NER'LLI TANKU
🎯 Referencia: ella no camina por el bosque.
🤖 Traducción: nosotros no camina por el bosque .

🎖️ BLEU: 75.98
🔸 Entrada:     KU' KENMA KU'LA WA'TEN'TULLI'DEK MUSICLANLEK'
🎯 Referencia: tú no espera con tranquilidad.
🤖 Traducción: ellos no espera con tranquilidad .

🎖️ BLEU: 75.98
🔸 Entrada:     KU' KENMA KU'LA LLINSERPI ALA'SA' KERKA'
🎯 Referencia: tú no lee un libro.
🤖 Tr